### Phase 2 - Feature Engineering

In [50]:
import pandas as pd
import numpy as np

In [51]:
df = pd.read_csv("../data/processed/eda_enriched.csv")

df = df.sort_values(["patient_id", "hour_from_admission"]).reset_index(drop=True)

df.head()

,hour_from_admission,heart_rate,respiratory_rate,spo2_pct,temperature_c,systolic_bp,diastolic_bp,oxygen_device,oxygen_flow,mobility_score,...,hemoglobin,sepsis_risk_score,age,gender,comorbidity_index,admission_type,deterioration_next_12h,patient_id,hr_std,NEWS2
0,0,68.58,14.47,96.52,37.18,108.94,78.43,none,0.0,2,...,13.55,0.2621,24,M,2,Elective,0,1,NaN,1
1,1,67.03,13.87,94.94,37.25,111.73,79.14,none,0.0,3,...,13.65,0.3353,24,M,2,Elective,0,1,NaN,1
2,2,69.05,14.63,94.45,37.29,111.48,78.86,none,0.0,2,...,13.69,0.1678,24,M,2,Elective,0,1,NaN,1
3,3,69.07,14.42,95.16,37.27,110.68,76.79,none,0.0,2,...,13.61,0.1961,24,M,2,Elective,0,1,NaN,0
4,4,73.35,15.62,95.83,37.21,110.38,75.47,none,0.0,3,...,13.49,0.3000,24,M,2,Elective,0,1,NaN,0


In [52]:
print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("\nMissing values:\n", df.isnull().sum())

Shape: (293248, 25)
Columns: 25

Missing values:
 hour_from_admission           0
heart_rate                    0
respiratory_rate              0
spo2_pct                      0
temperature_c                 0
systolic_bp                   0
diastolic_bp                  0
oxygen_device                 0
oxygen_flow                   0
mobility_score                0
nurse_alert                   0
wbc_count                     0
lactate                       0
creatinine                    0
crp_level                     0
hemoglobin                    0
sepsis_risk_score             0
age                           0
gender                        0
comorbidity_index             0
admission_type                0
deterioration_next_12h        0
patient_id                    0
hr_std                    35000
NEWS2                         0
dtype: int64


In [53]:
df["MAP"] = (df["systolic_bp"] + 2 * df["diastolic_bp"]) / 3

In [54]:
df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]

In [55]:
df["spo2_fio2"] = df["spo2_pct"] / (21 + 4 * df["oxygen_flow"])

In [56]:
vitals = ["heart_rate", "respiratory_rate", "spo2_pct", "temperature_c", "lactate"]

for col in vitals:
    df[f"{col}_roc"] = df.groupby("patient_id")[col].diff(2) / 2

In [57]:
for col in vitals:
    df[f"{col}_acc"] = df.groupby("patient_id")[f"{col}_roc"].diff()

In [58]:
WINDOW = 6

for col in vitals:
    rolling_mean = df.groupby("patient_id")[col].rolling(WINDOW).mean().reset_index(level=0, drop=True)
    rolling_std = df.groupby("patient_id")[col].rolling(WINDOW).std().reset_index(level=0, drop=True)

    df[f"{col}_roll_mean"] = rolling_mean
    df[f"{col}_roll_std"] = rolling_std
    df[f"{col}_cv"] = rolling_std / (rolling_mean + 1e-6)

In [59]:
df["tachycardia_flag"] = (df["heart_rate"] > 100).astype(int)

df["tachycardia_flag"] = (
    df.groupby("patient_id")["tachycardia_flag"]
    .rolling(3).sum().reset_index(level=0, drop=True) >= 3
).astype(int)

In [60]:
df["temp_diff"] = df.groupby("patient_id")["temperature_c"].diff()

df["fever_trend"] = (df["temp_diff"] > 0).astype(int)

df["fever_trend"] = (
    df.groupby("patient_id")["fever_trend"]
    .rolling(3).sum().reset_index(level=0, drop=True) >= 3
).astype(int)

In [61]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour_from_admission"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour_from_admission"] / 24)

In [62]:
df["time_bucket"] = pd.qcut(
    df["hour_from_admission"],
    q=5,
    labels=False,
    duplicates="drop"
)

In [63]:
def compute_news2(row):
    score = 0

    rr = row["respiratory_rate"]
    if rr <= 8 or rr >= 25: score += 3
    elif 21 <= rr <= 24: score += 2
    elif 9 <= rr <= 11: score += 1

    spo2 = row["spo2_pct"]
    if spo2 <= 91: score += 3
    elif spo2 <= 93: score += 2
    elif spo2 <= 95: score += 1

    temp = row["temperature_c"]
    if temp <= 35: score += 3
    elif temp >= 39.1: score += 2
    elif temp <= 36: score += 1

    sbp = row["systolic_bp"]
    if sbp <= 90: score += 3
    elif sbp <= 100: score += 2
    elif sbp <= 110: score += 1

    hr = row["heart_rate"]
    if hr <= 40 or hr >= 131: score += 3
    elif hr >= 111: score += 2
    elif hr >= 91: score += 1

    return score

In [64]:
df["NEWS2"] = df.apply(compute_news2, axis=1)

In [65]:
df["sofa_proxy"] = (
    df["lactate"] +
    df["creatinine"] +
    (100 - df["spo2_pct"]) +
    (100 - df["systolic_bp"])
)

In [66]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)

for col in categorical_cols:
    df[col] = df[col].astype("category").cat.codes

Categorical columns: ['oxygen_device', 'gender', 'admission_type']


In [67]:
df = df.fillna(method="ffill")
df = df.fillna(method="bfill")
df = df.fillna(0)

C:\Users\Swanandi\AppData\Local\Temp\ipykernel_11184\3528913340.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")
C:\Users\Swanandi\AppData\Local\Temp\ipykernel_11184\3528913340.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="bfill")


In [68]:
print(df.dtypes)

assert all(dtype != "object" for dtype in df.dtypes), "❌ Still contains non-numeric columns!"

hour_from_admission             int64
heart_rate                    float64
respiratory_rate              float64
spo2_pct                      float64
temperature_c                 float64
systolic_bp                   float64
diastolic_bp                  float64
oxygen_device                    int8
oxygen_flow                   float64
mobility_score                  int64
nurse_alert                     int64
wbc_count                     float64
lactate                       float64
creatinine                    float64
crp_level                     float64
hemoglobin                    float64
sepsis_risk_score             float64
age                             int64
gender                           int8
comorbidity_index               int64
admission_type                   int8
deterioration_next_12h          int64
patient_id                      int64
hr_std                        float64
NEWS2                           int64
MAP                           float64
pulse_pressu

In [69]:
df.to_csv("../data/features/features.csv", index=False)

print("✅ Saved to ../data/features/features.csv")

✅ Saved to ../data/features/features.csv
